## 1️⃣ Importation & Exploration

In [6]:
import pandas as pd
import seaborn as sns

df = pd.read_csv('bank.csv')

print("Taille du dataset :", df.shape)
print("\nInformations sur les colonnes :")
print(df.info())

print("\nValeurs manquantes :")
print(df.isnull().sum())

print("\nNombre de doublons :", df.duplicated(subset=['transaction_id']).sum())

Taille du dataset : (2060, 16)

Informations sur les colonnes :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2060 entries, 0 to 2059
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   transaction_id       2060 non-null   object 
 1   client_id            2060 non-null   object 
 2   date_transaction     2060 non-null   object 
 3   montant              2060 non-null   object 
 4   devise               2060 non-null   object 
 5   taux_change_eur      2060 non-null   float64
 6   montant_eur          2060 non-null   float64
 7   categorie            2060 non-null   object 
 8   produit              2060 non-null   object 
 9   agence               1996 non-null   object 
 10  type_operation       2060 non-null   object 
 11  statut               2060 non-null   object 
 12  score_credit_client  1893 non-null   float64
 13  segment_client       1955 non-null   object 
 14  solde_avant          206

## 2️⃣ Nettoyage des données

In [7]:
df = df.drop_duplicates(subset=['transaction_id'])

df['date_transaction'] = pd.to_datetime(df['date_transaction'], errors='coerce')

df['montant'] = df['montant'].astype(str).str.replace(',', '.').astype(float)

df['solde_avant'] = df['solde_avant'].astype(str).str.replace(' EUR', '').str.replace(',', '.').astype(float)

df['devise'] = df['devise'].str.upper()

df['segment_client'] = df['segment_client'].str.title()
df['agence'] = df['agence'].str.strip()

df['score_credit_client'] = df['score_credit_client'].fillna(df['score_credit_client'].median())
df['segment_client'] = df['segment_client'].fillna(df['segment_client'].mode()[0])
df['agence'] = df['agence'].fillna('Inconnue')

## 3️⃣ Détection & Traitement des Valeurs Aberrantes

In [8]:
Q1 = df['montant'].quantile(0.25)
Q3 = df['montant'].quantile(0.75)
IQR = Q3 - Q1
borne_inf = Q1 - 1.5 * IQR
borne_sup = Q3 + 1.5 * IQR

anomalie_montant = (df['montant'] < borne_inf) | (df['montant'] > borne_sup)
anomalie_score = (df['score_credit_client'] < 0) | (df['score_credit_client'] > 850)

df['is_anomaly'] = anomalie_montant | anomalie_score

print("Nombre d'anomalies détectées :", df['is_anomaly'].sum())

Nombre d'anomalies détectées : 112


## 4️⃣ Feature Engineering

In [10]:
df['annee'] = df['date_transaction'].dt.year
df['mois'] = df['date_transaction'].dt.month
df['trimestre'] = df['date_transaction'].dt.quarter
df['jour_semaine'] = df['date_transaction'].dt.day_name()

df['montant_eur_verifie'] = df['montant'] / df['taux_change_eur']

def risque(score):
    if score >= 700: return 'Low'
    elif score >= 580: return 'Medium'
    else: return 'High'

df['categorie_risque'] = df['score_credit_client'].apply(risque)

stats_client = df.groupby('client_id').agg(
    nb_transactions=('transaction_id', 'count'),
    montant_moyen=('montant_eur', 'mean'),
    nb_produits=('produit', 'nunique')
).reset_index()

df = df.merge(stats_client, on='client_id', how='left')

## 5️⃣ Export & Documentation

In [11]:
df.to_csv('financecore_clean.csv', index=False)